In [1]:
import os
import json
import time
import re
import pandas as pd
import concurrent.futures
import httpx
from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(dotenv_path='/home/check/DATA/university/yr3, hk2/IE403 - Khai thác dữ liệu truyền thông xã hội/API.env')

True

In [2]:
TOGETHER_API_KEY = os.getenv("Together_AI_API") 

client = OpenAI(
    api_key=TOGETHER_API_KEY,
    base_url="https://api.together.xyz/v1",
    timeout=httpx.Timeout(120.0) 
)

# Chuyển đổi sang Model Flagship mạnh nhất của nhà Qwen
MODEL_JUDGE = "Qwen/Qwen3-235B-A22B-Instruct-2507-tput"

In [3]:
# ==========================================
# 2. HÀM GỌI API GÁN NHÃN (ĐÃ FIX LỖI STREAM CHUNK RỖNG)
# ==========================================
def get_sarcasm_labels_qwen(batch_data):
    system_prompt = """Bạn là một CHUYÊN GIA NGÔN NGỮ HỌC và HỆ THỐNG ĐÁNH GIÁ (LLM-as-a-Judge) cấp cao.
Nhiệm vụ của bạn là gán nhãn "Sarcastic" (Mỉa mai) hoặc "Non-Sarcastic" (Không mỉa mai) cho các bình luận. Đồng thời, NẾU bình luận là "Sarcastic", bạn BẮT BUỘC phải phân loại nhóm mỉa mai (sarcasm_type).

<sarcasm_guideline>
<data_priority_rule>
LƯU Ý VỀ ĐỘ TIN CẬY CỦA DỮ LIỆU:
1. [Context] (Ngữ cảnh) và [Comment] (Bình luận) là sự thật tuyệt đối, BẮT BUỘC phải dựa vào đây để đánh giá.
2. [Commonsense] (Kiến thức nền) CHỈ LÀ THÔNG TIN THAM KHẢO để hỗ trợ giải nghĩa từ lóng. Nếu bạn phát hiện [Commonsense] có dấu hiệu ngây ngô, suy diễn sai lệch, hoặc mâu thuẫn với logic hiển nhiên của [Context], HÃY NGAY LẬP TỨC BỎ QUA [Commonsense] và tự suy luận dựa trên [Context].
</data_priority_rule>

ĐỊNH NGHĨA HỌC THUẬT VỀ SARCASM (MỈA MAI):
Mỉa mai là một hình thức châm biếm bằng lời nói (verbal irony), thường dùng để chế nhạo, chỉ trích hoặc thể hiện sự khinh miệt một cách gián tiếp. 

Đặc trưng cốt lõi của Mỉa mai là "SỰ MÂU THUẪN" (Incongruity). Hãy tìm kiếm 3 dấu hiệu sau:
1. Mâu thuẫn với Ngữ cảnh (Contextual Incongruity): Ý nghĩa bề mặt của bình luận (thường là tích cực) trái ngược hoàn toàn với sự thật hiển nhiên được cung cấp trong [Context] và [Commonsense].
2. Lời khen giả tạo (False Praise): Khen ngợi một người/sự kiện thực chất đang rất tồi tệ, thất bại hoặc đầy tai tiếng.
3. Vi phạm kỳ vọng thực tế (Pragmatic Expectation Violation): Dùng một hình ảnh so sánh tâng bốc (Hyperbole) đặt cạnh một đối tượng có đặc điểm hoàn toàn trái ngược trong đời thực.

ĐẶC TRƯNG "ĐÁ XÉO / CÀ KHỊA" TRONG TIẾNG VIỆT:
Ngoài định nghĩa cốt lõi, hãy đặc biệt chú ý 3 hình thái mỉa mai ngầm ẩn (đá xéo) cực kỳ phổ biến sau:
1. Lịch sự giả tạo (Mock Politeness / Nói mát): Dùng từ ngữ nhún nhường, xưng hô trang trọng hoặc khen ngợi thái quá ("dạ vâng", "anh thì nhất rồi", "giỏi ghê ha") đối với một hành vi sai trái/người có lỗi. -> Sarcastic.
2. Khen ngợi lạc đề / Hiển nhiên (Irrelevant Praise): Khen ngợi một điều quá đỗi hiển nhiên, lặt vặt (VD: "Ít ra anh ấy mặc áo đẹp") khi bối cảnh đang đề cập đến một thất bại lớn về chuyên môn, nhằm hạ thấp năng lực thực sự. -> Sarcastic.
3. Dấu hiệu ngôn ngữ tình thái: Các từ cuối câu như "nhỉ", "ghê", "ha", "cơ", "á", "thay" khi đặt cạnh một tính từ tích cực trong một hoàn cảnh tiêu cực. -> Sarcastic.

PHÂN LOẠI 5 NHÁNH MỈA MAI (SARCASM TYPES):
Nếu xác định là Sarcastic, hãy xếp nó vào 1 trong 5 loại sau:
1. "Lexical Contradiction" (Mâu thuẫn từ vựng): Thường áp dụng cho "Lời khen giả tạo" hoặc "Dấu hiệu ngôn ngữ tình thái".
2. "Propositional Contradiction" (Mâu thuẫn mệnh đề): Thường áp dụng cho "Lịch sự giả tạo" hoặc "Khen ngợi lạc đề".
3. "Rhetorical Question" (Câu hỏi tu từ): Các câu hỏi mang tính thách thức, ngầm phủ định (VD: "Làm như hay lắm không bằng?").
4. "Wh-Question" (Câu hỏi nghi vấn Wh-): Dùng từ để hỏi (Ai, Cái gì, Ở đâu, Sao) để móc mỉa (VD: "Ủa rồi mua bằng ở đâu vậy?").
5. "Hypothetical" (Giả định): Áp dụng cho "Vi phạm kỳ vọng thực tế", đưa ra giả định phóng đại, phi logic (VD: "Nó mà giỏi thì tao đi đầu xuống đất").

HƯỚNG DẪN CHẤM ĐIỂM (REASONING CHAIN):
- NẾU: Bình luận chứa lời khen/đồng tình + [Context/Thực tế] xác nhận thực tế là tồi tệ/thất bại => Sarcastic (Type: Lexical hoặc Propositional).
- NẾU: Bình luận là câu hỏi tu từ mang tính thách thức, ngầm phủ định sự thật => Sarcastic (Type: Rhetorical hoặc Wh-Question).
- NẾU: Bình luận dùng thành ngữ/tục ngữ kết hợp với lời tâng bốc thái quá, phi lý so với thực tế => Sarcastic (Type: Hypothetical).
- NẾU: Bình luận dùng thành ngữ/ẩn dụ để chê bai TRỰC TIẾP, không mượn vỏ bọc khen ngợi => Non-Sarcastic.
- NẾU: Bình luận chê bai/chửi bới trực tiếp, khen ngợi đúng với sự thật hiển nhiên, hoặc chỉ là câu hỏi thắc mắc tìm kiếm thông tin thật => Non-Sarcastic.
</sarcasm_guideline>

<examples>
HÃY HỌC TẬP CÁCH PHÂN TÍCH QUA CÁC VÍ DỤ SAU:
- [Context]: Một người lái xe ẩu gây tai nạn. | [Comment]: "Trình độ lái xe điêu luyện quá cơ, nhắm mắt cũng đâm trúng."
  -> Nhãn: Sarcastic | Type: Lexical Contradiction. (Reasoning: Dùng từ khen ngợi "điêu luyện" + từ tình thái "quá cơ" mâu thuẫn trực tiếp với hành vi tai nạn tồi tệ).
  
- [Context]: Một bộ phim bị giới phê bình chê tơi tả về nội dung. | [Comment]: "Tuyệt vời, ít ra diễn viên có mặc quần áo."
  -> Nhãn: Sarcastic | Type: Propositional Contradiction. (Reasoning: Lời khen giả tạo, khen ngợi một đặc điểm lặt vặt hiển nhiên để ngầm chê nội dung phim).

- [Context]: Một dự án làm 10 năm chưa xong. | [Comment]: "Đợi xong cái này chắc tôi xuống mồ."
  -> Nhãn: Sarcastic | Type: Hypothetical. (Reasoning: Đưa ra một giả định phóng đại về cái chết để châm biếm tiến độ quá chậm).

- [Context]: Cầu thủ A sút hỏng phạt đền. | [Comment]: "Đúng là thằng phế vật, sút ngu như bò."
  -> Nhãn: Non-Sarcastic | Type: None. (Reasoning: Chửi bới, chê bai trực tiếp không có lớp vỏ bọc ẩn ý).
</examples>

<output_format>
BẠN BẮT BUỘC TRẢ VỀ JSON CHỨA MẢNG "results". MỖI PHẦN TỬ GỒM:
- "id": Trích xuất từ Input.
- "comment": Trích xuất nguyên văn bình luận để làm mỏ neo.
- "sarcasm_label": BẮT BUỘC LÀ "Sarcastic" HOẶC "Non-Sarcastic".
- "sarcasm_type": Chọn 1 trong 5 loại ["Lexical Contradiction", "Propositional Contradiction", "Rhetorical Question", "Wh-Question", "Hypothetical"]. NẾU label là "Non-Sarcastic", điền là "None".
- "reasoning": 1 câu giải thích ngắn gọn (BẮT BUỘC DƯỚI 40 CHỮ) về sự mâu thuẫn giữa Comment và Ngữ cảnh (Context) để tiết kiệm token đầu ra.
</output_format>

BÂY GIỜ, HÃY PHÂN TÍCH VÀ GÁN NHÃN CHO BATCH DỮ LIỆU SAU:
"""

    user_prompt = "[Input Batch]\n"
    for item in batch_data:
        user_prompt += f"ID: {item['id']} | Context: {item['ctx']} | Comment: {item['cmt']} | Commonsense: {item['cs']}\n"
    user_prompt += "\n[Output (JSON)]:"

    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=MODEL_JUDGE,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.0,
                response_format={"type": "json_object"},
                max_tokens=4096,
                stream=False
            )
            
            content = response.choices[0].message.content or ""
                        
            if not content: 
                tqdm.write(f"⚠️ Batch rỗng nội dung (Lần {attempt+1}/3)")
                time.sleep(2)
                continue
                
            try:
                # Xóa các ký tự ẩn gây lỗi JSON
                content = re.sub(r'[\x00-\x1f]', '', content)
                result_json = json.loads(content, strict=False)
                
                if isinstance(result_json, list):
                    return result_json
                # Nếu Qwen trả về Dict (có bọc "results") -> Dùng .get()
                elif isinstance(result_json, dict):
                    return result_json.get("results", [])
                else:
                    return None
                    
            except json.JSONDecodeError as e:
                tqdm.write(f"⚠️ Lỗi parse JSON (Lần {attempt+1}/3): {e}. Đầu output: {content[:300]}")
                time.sleep(2)
                continue

        except Exception as e:
            tqdm.write(f"⚠️ Lỗi API Qwen Max (Lần {attempt+1}/3): {str(e)}")
            time.sleep(4) 
            
    return None

In [4]:
def process_single_batch(batch_info):
    batch_idx, batch_data = batch_info
    results = get_sarcasm_labels_qwen(batch_data)
    return batch_idx, results

In [5]:
def process_dataset_labeling(input_csv, output_csv, batch_size=15, max_workers=3):
    if os.path.exists(output_csv):
        print(f"\n[*] Tìm thấy file đang chạy dở: {output_csv}. Đang resume...")
        df = pd.read_csv(output_csv)
    else:
        print(f"\n[*] Chạy lần đầu. Đang đọc file gốc: {input_csv}")
        df = pd.read_csv(input_csv)
        df['sarcasm_label'] = None 
        df['sarcasm_type'] = None
        df['reasoning'] = None
        df.to_csv(output_csv, index=False, encoding='utf-8-sig')
    
    if 'sarcasm_label' not in df.columns: df['sarcasm_label'] = None
    if 'sarcasm_type' not in df.columns: df['sarcasm_type'] = None
    if 'reasoning' not in df.columns: df['reasoning'] = None

    def needs_processing(x):
        return str(x).strip() in ['nan', 'None', '']

    pending_indices = df[df['sarcasm_label'].apply(needs_processing)].index.tolist()
    print(f"Tổng số dòng chờ Qwen Max gán nhãn: {len(pending_indices)} | Batch: {batch_size} | Luồng: {max_workers}")
    
    if not pending_indices:
        print("🎉 Mọi dữ liệu đã được gán nhãn hoàn tất 100%!")
        return

    # Chia mảng dữ liệu thành các cục (Batches)
    all_batches = []
    for i in range(0, len(pending_indices), batch_size):
        batch_indices = pending_indices[i:i+batch_size]
        batch_data = []
        for idx in batch_indices:
            ctx = str(df.at[idx, 'video_core_content']).replace('\n', ' ')
            cmt = str(df.at[idx, 'comment']).replace('\n', ' ')
            cs = str(df.at[idx, 'commonsense']).replace('\n', ' ')
            
            if cmt.lower() == 'nan' or cs.lower() == 'nan': 
                continue
                
            batch_data.append({"id": idx, "ctx": ctx, "cmt": cmt, "cs": cs})
        
        if batch_data:
            all_batches.append((i // batch_size, batch_data))

    print(f"🚀 Kích hoạt cỗ máy Giám khảo siêu cấp Qwen Max...")

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(process_single_batch, b) for b in all_batches]
        
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Đang Judge (Qwen Max)"):
            batch_idx, results = future.result()
            
            if results:
                for res in results:
                    try:
                        row_id = int(res.get("id"))
                    except (TypeError, ValueError):
                        tqdm.write(f"⚠️ Bỏ qua kết quả có id không hợp lệ: {res}")
                        continue

                    label = res.get("sarcasm_label", "")
                    sarcasm_type = res.get("sarcasm_type", "")
                    reason = res.get("reasoning", "")

                    if label not in ["Sarcastic", "Non-Sarcastic"]:
                        tqdm.write(f"⚠️ Bỏ qua nhãn không hợp lệ tại dòng {row_id}: {label}")
                        continue
                        
                    if row_id is not None and row_id in df.index:
                        df.at[row_id, 'sarcasm_label'] = str(label).strip()
                        df.at[row_id, 'sarcasm_type'] = str(sarcasm_type).strip()
                        df.at[row_id, 'reasoning'] = str(reason).strip()
            
            # Lưu đè liên tục để bảo vệ dữ liệu
            df.to_csv(output_csv, index=False, encoding='utf-8-sig')

    print(f"\n[HOÀN THÀNH TẬP GOLD DATA] Toàn bộ dữ liệu nhãn chuẩn đã lưu tại: {output_csv}")

In [6]:
if __name__ == "__main__":
    INPUT_FILE = "/home/check/DATA/university/yr3, hk2/IE403 - Khai thác dữ liệu truyền thông xã hội/Project/data/gold/test_data_infer.csv" 
    OUTPUT_FILE = "Qwen3_A22B_combine.csv" 
    
    BATCH_SIZE = 10 
    MAX_WORKERS = 5
    
    if os.path.exists(INPUT_FILE):
        process_dataset_labeling(INPUT_FILE, OUTPUT_FILE, batch_size=BATCH_SIZE, max_workers=MAX_WORKERS)
    else:
        print(f"Lỗi: Không tìm thấy file gốc {INPUT_FILE}")


[*] Tìm thấy file đang chạy dở: Qwen3_A22B_combine.csv. Đang resume...
Tổng số dòng chờ Qwen Max gán nhãn: 747 | Batch: 10 | Luồng: 5
🚀 Kích hoạt cỗ máy Giám khảo siêu cấp Qwen Max...


Đang Judge (Qwen Max): 100%|██████████| 75/75 [09:14<00:00,  7.39s/it]


[HOÀN THÀNH TẬP GOLD DATA] Toàn bộ dữ liệu nhãn chuẩn đã lưu tại: Qwen3_A22B_combine.csv
